In [ ]:
%pip install -e ..
# Editable install of this branch (not upstream master) -- preview_pipeline() is new,
# unreleased code.


In [ ]:
# Download the models needed to resolve the pipeline, plus the patient-UnitTest2 test
# data (same patient as 03_run_postoperative_segmentation.ipynb). The patient data is
# only used for the real baseline run below -- preview_pipeline() itself never reads
# any scan.
import os
import requests
import zipfile

patient_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Samples-RaidionicsRADSLib-UnitTest2.zip'
brain_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_Brain-v13.zip'
seq_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_SequenceClassifier-v13.zip'
rest_tumor_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_TumorCE_Postop-v13.zip'
cavity_model_url = 'https://github.com/raidionics/Raidionics-models/releases/download/v1.3.0-rc/Raidionics-MRI_Cavity-v13.zip'

test_dir = os.path.join(os.getcwd(), 'unit_tests_results_dir')
patient_dir = os.path.join(test_dir, 'patient')
models_dir = os.path.join(test_dir, 'models')
results_dir = os.path.join(test_dir, 'results')
os.makedirs(patient_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

archive_dl_dest = os.path.join(test_dir, 'inference_patient.zip')
if not os.path.exists(archive_dl_dest):
    response = requests.get(patient_url, stream=True)
    response.raise_for_status()
    with open(archive_dl_dest, "wb") as f:
        for chunk in response.iter_content(chunk_size=1048576):
            f.write(chunk)
with zipfile.ZipFile(archive_dl_dest, 'r') as zip_ref:
    zip_ref.extractall(patient_dir)

for archive_name, url in [('seq-model.zip', seq_model_url),
                          ('brain-model.zip', brain_model_url),
                          ('rest_tumor-model.zip', rest_tumor_model_url),
                          ('cavity-model.zip', cavity_model_url)]:
    archive_dl_dest = os.path.join(test_dir, archive_name)
    if not os.path.exists(archive_dl_dest):
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(archive_dl_dest, "wb") as f:
            for chunk in response.iter_content(chunk_size=1048576):
                f.write(chunk)
    with zipfile.ZipFile(archive_dl_dest, 'r') as zip_ref:
        zip_ref.extractall(models_dir)


In [ ]:
# Prepare the pipeline -- same pipeline.json as for an actual run. The preview only
# resolves which task/model each step refers to, it never runs any of them.
import json

pipeline_dir = os.path.join(test_dir, 'pipelines')
os.makedirs(pipeline_dir, exist_ok=True)

pipeline_json = {}
step_index = 1
step_str = str(step_index)
pipeline_json[step_str] = {}
pipeline_json[step_str]["task"] = "Classification"
pipeline_json[step_str]["inputs"] = {}  # Empty input means running it on all existing data for the patient
pipeline_json[step_str]["target"] = ["MRSequence"]
pipeline_json[step_str]["model"] = "MRI_SequenceClassifier"
pipeline_json[step_str]["description"] = "Classification of the MRI sequence type for all input scans."

step_index = step_index + 1
step_str = str(step_index)
pipeline_json[step_str] = {}
pipeline_json[step_str]["task"] = 'Model selection'  # Will select the appropriate model for the provided set of MR scans
pipeline_json[step_str]["model"] = 'MRI_TumorCE_Postop'
pipeline_json[step_str]["timestamp"] = 1  # Timestamp 1 indicates early post-operative data, located in folder T1
pipeline_json[step_str]["format"] = "thresholding"
pipeline_json[step_str]["description"] = "Identifying the best rest tumor segmentation model for existing inputs"

step_index = step_index + 1
step_str = str(step_index)
pipeline_json[step_str] = {}
pipeline_json[step_str]["task"] = 'Model selection'
pipeline_json[step_str]["model"] = 'MRI_Cavity'
pipeline_json[step_str]["timestamp"] = 1
pipeline_json[step_str]["format"] = "thresholding"
pipeline_json[step_str]["description"] = "Identifying the best cavity segmentation model for existing inputs"

print(json.dumps(pipeline_json, indent=4))
with open(os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'), 'w', newline='\n') as outfile:
    json.dump(pipeline_json, outfile, indent=4, sort_keys=True)

In [ ]:
# Run the real pipeline once on patient-UnitTest2, to get a genuine "already computed"
# baseline for the check further below. The only cell in this notebook that performs
# real computation (a few minutes on CPU).
import configparser
from raidionicsrads.compute import run_rads

baseline_results_dir = os.path.join(results_dir, "output_postop_segmentation_baseline")
os.makedirs(baseline_results_dir, exist_ok=True)

baseline_config = configparser.ConfigParser()
baseline_config.add_section('Default')
baseline_config.set('Default', 'task', 'neuro_diagnosis')
baseline_config.set('Default', 'caller', '')
baseline_config.add_section('System')
baseline_config.set('System', 'gpu_id', '-1')
baseline_config.set('System', 'input_folder', os.path.join(patient_dir, 'patient-UnitTest2', 'inputs'))
baseline_config.set('System', 'output_folder', baseline_results_dir)
baseline_config.set('System', 'model_folder', models_dir)
baseline_config.set('System', 'pipeline_filename', os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'))
baseline_config.add_section('Runtime')
baseline_config.set('Runtime', 'reconstruction_method', 'thresholding')
baseline_config.set('Runtime', 'reconstruction_order', 'resample_first')
baseline_config.set('Runtime', 'use_stripped_data', 'True')
baseline_config.set('Runtime', 'use_registered_data', 'False')

baseline_config_filename = os.path.join(baseline_results_dir, 'rads_config.ini')
with open(baseline_config_filename, 'w') as outfile:
    baseline_config.write(outfile)

run_rads(baseline_config_filename)


In [ ]:
# Declare the MR sequences known to be available, per timestamp, instead of pointing
# to real image files. T0 is the preoperative timestamp, T1 the early postoperative one.
# (This matches what patient-UnitTest2, used in 03_run_postoperative_segmentation.ipynb,
# actually contains -- T1-CE only preop, T1-CE/T1-w/FLAIR postop -- so the resolved
# pipeline below can be compared against a real run of the same data.)
sequences_declaration = {
    "T0": ["T1-CE"],
    "T1": ["T1-CE", "T1-w", "FLAIR"]
}

# Prepare the configuration file -- input_folder is left empty since no real patient
# data is read for a preview.
import configparser
import logging

test_results_dir = os.path.join(results_dir, "output_preview_postop_segmentation")
os.makedirs(test_results_dir, exist_ok=True)

logging.basicConfig()
logging.getLogger().setLevel(logging.INFO)
rads_config = configparser.ConfigParser()
rads_config.add_section('Default')
rads_config.set('Default', 'task', 'neuro_diagnosis')
rads_config.set('Default', 'caller', '')
rads_config.add_section('System')
rads_config.set('System', 'gpu_id', "-1")
rads_config.set('System', 'input_folder', '')
rads_config.set('System', 'output_folder', test_results_dir)
rads_config.set('System', 'model_folder', models_dir)
rads_config.set('System', 'pipeline_filename', os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'))
rads_config.add_section('Runtime')
rads_config.set('Runtime', 'reconstruction_method', 'thresholding')
rads_config.set('Runtime', 'reconstruction_order', 'resample_first')
rads_config.set('Runtime', 'use_stripped_data', 'True')
rads_config.set('Runtime', 'use_registered_data', 'False')

rads_config_filename = os.path.join(test_results_dir, 'rads_config.ini')
with open(rads_config_filename, 'w') as outfile:
    rads_config.write(outfile)

In [ ]:
# Build executed_pipeline.json from the declared sequences -- setup() only, nothing
# is actually computed.
from raidionicsrads.compute import preview_pipeline

executed_pipeline = preview_pipeline(config_filename=rads_config_filename,
                                     sequences_declaration=sequences_declaration)
print(json.dumps(executed_pipeline, indent=4))

In [ ]:
# Inspecting what actually landed on disk: only executed_pipeline.json plus empty
# per-timestamp scaffolding folders should be there -- no predictions, no working
# directories, since nothing was executed.
for root, dirs, files in os.walk(test_results_dir):
    level = root.replace(test_results_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or os.path.basename(test_results_dir)}/")
    for f in sorted(files):
        print(f"{indent}  {f}")

In [ ]:
# Sanity check: declaring fewer sequences (only T1-CE, no T1-w/FLAIR at T1) should
# make model selection resolve to the lighter "t1c"-only submodels instead of the
# "t1c_t1w_t2f_t1d" ones picked above -- demonstrates the preview reacts correctly to
# what is declared as available.
sparse_sequences_declaration = {
    "T0": ["T1-CE"],
    "T1": ["T1-CE"]
}

sparse_results_dir = os.path.join(results_dir, "output_preview_postop_segmentation_sparse")
os.makedirs(sparse_results_dir, exist_ok=True)
rads_config.set('System', 'output_folder', sparse_results_dir)
sparse_config_filename = os.path.join(sparse_results_dir, 'rads_config.ini')
with open(sparse_config_filename, 'w') as outfile:
    rads_config.write(outfile)

sparse_executed_pipeline = preview_pipeline(config_filename=sparse_config_filename,
                                            sequences_declaration=sparse_sequences_declaration)
for step in sparse_executed_pipeline.values():
    if step.get("task") == "Segmentation":
        print(step["model"], "--", step["description"])

In [ ]:
# Checks whether a resolved pipeline step's output already exists, given lookup
# tables of already-computed targets/registrations. Used by the real-data derivation
# below.

def would_step_be_skipped(step, already_computed_targets, already_computed_registrations):
    task = step.get("task")

    if task in ("Segmentation", "Segmentation refinement"):
        target = step.get("target")
        if not target:
            return None
        ts = step["inputs"]["0"]["timestamp"] if step.get("inputs") else step.get("timestamp")
        have = already_computed_targets.get(ts, set())
        return all(t in have for t in target)

    if task in ("Registration", "Apply registration"):
        moving, fixed = step["moving"], step["fixed"]
        key = (moving["sequence"], moving["timestamp"], fixed["sequence"], fixed["timestamp"])
        return key in already_computed_registrations

    return None  # Classification / Model selection / Reporting selection


status_label = {True: "ALREADY COMPUTED -- can be skipped",
                False: "MISSING -- must run",
                None: "(not relevant for this check)"}


In [ ]:
# Build a real input folder (raw MR volumes + already-computed results from the
# baseline run above) and read it with the real PatientParameters, to get genuine
# already_computed_targets/already_computed_registrations instead of made-up data.
#
# This models direct raidionics_rads_lib reuse (caller='raidionics'), not the
# Raidionics GUI's own mechanism -- the GUI uses a different annotation naming
# convention and does not restage registrations at all.
#
# Annotation filenames are renamed here to drop the model-name suffix
# (e.g. "_MRI_Cavity"), since the reader only understands the plain form.
import shutil
from raidionicsrads.Utils.DataStructures.PatientStructure import PatientParameters
from raidionicsrads.Utils.configuration_parser import ResourcesConfiguration

patient_source_dir = os.path.join(patient_dir, 'patient-UnitTest2', 'inputs')

merged_dir = os.path.join(test_dir, 'merged_input_for_reuse_check')
if os.path.exists(merged_dir):
    shutil.rmtree(merged_dir)
os.makedirs(os.path.join(merged_dir, 'T0', 'raw'))
os.makedirs(os.path.join(merged_dir, 'T1', 'raw'))

# Raw MR volumes
shutil.copy(os.path.join(patient_source_dir, 'T0', 'preop_t1gd.nii.gz'), os.path.join(merged_dir, 'T0', 'raw'))
for f in ['postop_flair.nii.gz', 'postop_t1.nii.gz', 'postop_t1gd.nii.gz']:
    shutil.copy(os.path.join(patient_source_dir, 'T1', f), os.path.join(merged_dir, 'T1', 'raw'))

# Already-computed annotations, renamed to drop the model-name suffix.
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1gd_annotation-Cavity_MRI_Cavity.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1gd_annotation-Cavity.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1gd_annotation-TumorCE_MRI_TumorCE_Postop.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1gd_annotation-TumorCE.nii.gz'))

# Brain masks use a different naming convention on disk ("_label_Brain.nii.gz");
# renamed to match the others.
shutil.copy(os.path.join(baseline_results_dir, 'T0', 'preop_t1gd_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T0', 'raw', 'preop_t1gd_annotation-Brain.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_flair_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_flair_annotation-Brain.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1_annotation-Brain.nii.gz'))
shutil.copy(os.path.join(baseline_results_dir, 'T1', 'postop_t1gd_label_Brain.nii.gz'),
           os.path.join(merged_dir, 'T1', 'raw', 'postop_t1gd_annotation-Brain.nii.gz'))

# Registration results, copied as-is.
shutil.copytree(os.path.join(baseline_results_dir, 'T1', 'T1_T1c_space'),
                os.path.join(merged_dir, 'T1', 'raw', 'T1_T1c_space'))

# caller='raidionics' is required for the raw/ subfolder and annotation-filename
# convention used above.
merged_config = configparser.ConfigParser()
merged_config.add_section('Default')
merged_config.set('Default', 'task', 'neuro_diagnosis')
merged_config.set('Default', 'caller', 'raidionics')
merged_config.add_section('System')
merged_config.set('System', 'gpu_id', '-1')
merged_config.set('System', 'input_folder', merged_dir)
merged_config.set('System', 'output_folder', os.path.join(results_dir, 'merged_input_scratch'))
merged_config.set('System', 'model_folder', models_dir)
merged_config.set('System', 'pipeline_filename', os.path.join(pipeline_dir, 'pipeline_postop_segmentation.json'))
merged_config.add_section('Runtime')
merged_config.set('Runtime', 'reconstruction_method', 'thresholding')
merged_config.set('Runtime', 'reconstruction_order', 'resample_first')
merged_config.set('Runtime', 'use_stripped_data', 'False')
merged_config.set('Runtime', 'use_registered_data', 'False')
os.makedirs(os.path.join(results_dir, 'merged_input_scratch'), exist_ok=True)
merged_config_filename = os.path.join(merged_dir, 'rads_config.ini')
with open(merged_config_filename, 'w') as outfile:
    merged_config.write(outfile)

ResourcesConfiguration.getInstance().set_environment(config_path=merged_config_filename)
patient_parameters = PatientParameters(id="Patient", patient_filepath=merged_dir)

print("Real annotations found by PatientParameters:")
already_computed_targets = {}
for anno_uid, anno in patient_parameters.annotation_volumes.items():
    vol = patient_parameters.get_radiological_volume(anno._radiological_volume_uid)
    ts = int(vol._timestamp_id[1:])  # "T1" -> 1
    already_computed_targets.setdefault(ts, set()).add(anno._annotation_type.name)
    print(f"  {anno_uid}: {anno.get_annotation_type_str()} @ T{ts} (volum: {vol.get_sequence_type_str()})")

print()
print("Real registrations found by PatientParameters:")
already_computed_registrations = set()
for vol_uid, vol in patient_parameters.radiological_volumes.items():
    for dest_uid in vol.get_registered_volume_destination_uids():
        fixed_vol = patient_parameters.get_radiological_volume(dest_uid)
        if fixed_vol is None:
            continue
        moving_ts = int(vol._timestamp_id[1:])
        fixed_ts = int(fixed_vol._timestamp_id[1:])
        key = (vol.get_sequence_type_str(), moving_ts, fixed_vol.get_sequence_type_str(), fixed_ts)
        already_computed_registrations.add(key)
        print(f"  {vol.get_sequence_type_str()} @ T{moving_ts} -> {fixed_vol.get_sequence_type_str()} @ T{fixed_ts}")

print()

for k in sorted(executed_pipeline.keys(), key=int):
    step = executed_pipeline[k]
    status = would_step_be_skipped(step, already_computed_targets, already_computed_registrations)
    if status is not None:
        print(f"{k:>2} {step.get('task'):<24} target={str(step.get('target')):<12} -> {status_label[status]}")
